In [1]:
import json
from pathlib import Path
import google.generativeai as genai

c:\Users\tientm1\DS300-UIT-RecommenderSystem\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\tientm1\AppData\Local\Temp\ipykernel_12192\4199492748.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


# RA-Rec: Conversational Recommender System for Food

This notebook implements a prompt-driven dialogue state tracking loop based on the RA-Rec framework.

**System Flow:**
1. User Utterance → Prompt-based Intent Classification
2. Intent Classification → Prompt-based State Update
3. State Update → Action Selection
4. Action Selection → Response Generation

**User Intents:** Provide Preference, Inquire, Accept Recommendation, Reject Recommendation

**System Actions:** Request Information, Recommend and Explain, Answer, Respond to Acceptance, Respond to Rejection

---

## 1. Configuration

In [ ]:
# LLM Configuration
import os
from dotenv import load_dotenv
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)

MODEL = "models/gemini-2.5-flash-lite"
STATE_FILE = "state.json"

# Default state structure - tracks user preferences and dialogue history
DEFAULT_STATE = {
    "hard_constraints": {
        "type_of_food": [],   # Type of dish (e.g., "món Tết", "món chính")
        "ingredients": [],    # Main ingredients (e.g., "thịt lợn", "hải sản")
    },
    "soft_constraints": {
        "cook_time": [],      # Cooking time preference
        "num_of_people": [],  # Number of servings
        "calories": [],       # Calorie level preference
        "algeric": []         # Allergies
    },
    "recommended_items": [],  # List of recommended dishes
    "accepted_items": [],     # Dishes user accepted/liked
    "rejected_items": []      # Dishes user rejected/disliked
}

✅ Configuration loaded


In [ ]:
# State management functions
def load_state():
    """Load dialogue state from JSON file"""
    if Path(STATE_FILE).exists():
        return json.load(open(STATE_FILE, "r", encoding="utf-8"))
    else:
        json.dump(DEFAULT_STATE, open(STATE_FILE, "w", encoding="utf-8"), indent=4, ensure_ascii=False)
        return DEFAULT_STATE

def save_state(state):
    """Save dialogue state to JSON file"""
    json.dump(state, open(STATE_FILE, "w", encoding="utf-8"), indent=4, ensure_ascii=False)

✅ State management functions defined


In [ ]:
# STEP 1: PROMPT-BASED INTENT CLASSIFICATION

def classify_intent(user_utterance):
    prompt = f"""
            Classify the USER INTENT in a conversational recommender system.

            Possible intents (can be multiple):
            - "Provide Preference" - user states what they want (food type, ingredients, cooking time, servings, calories)
            - "Inquire" - user asks questions
            - "Accept Recommendation" - user accepts/likes a recommended dish
            - "Reject Recommendation" - user rejects/dislikes a recommended dish

            Examples (in Vietnamese):
            - "Tôi muốn tìm món Tết" → ["Provide Preference"]
            - "Món nào nấu nhanh cho 2 người" → ["Provide Preference"]
            - "Bạn gợi ý món gì?" → ["Inquire"]
            - "Món này hay đấy" → ["Accept Recommendation"]
            - "Không, tôi không thích món này" → ["Reject Recommendation"]
            - "Tôi thích món này" → ["Accept Recommendation"]
            - "Cho tôi món khác" → ["Reject Recommendation"]

            User says: "{user_utterance}"

            Return ONLY a JSON array of intent strings (e.g., ["Provide Preference"]).
            """
    
    resp = genai.GenerativeModel(MODEL).generate_content(prompt)
    
    try:
        text = resp.text.strip()
        # Remove markdown code blocks if present
        if "```" in text:
            parts = text.split("```")
            if len(parts) >= 2:
                text = parts[1]
                if text.startswith("json"):
                    text = text[4:]
                text = text.strip()
        
        text = text.strip()
        return json.loads(text)
    except Exception as e:
        print(f"❌ Error parsing intent JSON: {e}")
        print(f"Response: {resp.text}")
        return []

✅ Intent classification function defined


In [5]:
# Test step 1
test_intent = classify_intent("Tôi muốn tìm món Tết")  # Expected: ["Provide Preference"]
test_intent

['Provide Preference']

In [ ]:
# STEP 2: PROMPT-BASED STATE UPDATE

def update_state(user_utterance, intents, state):
    """
    Update dialogue state based on user utterance and classified intents using LLM.
    
    Args:
        user_utterance (str): User's input text
        intents (list): Classified intents
        state (dict): Current dialogue state
        
    Returns:
        dict: Updated dialogue state
    """
    
    # Handle "Provide Preference" intent
    if "Provide Preference" in intents:
        # Detect if user is providing complete info upfront
        type_of_food_empty = len(state["hard_constraints"]["type_of_food"]) == 0
        ingredients_empty = len(state["hard_constraints"]["ingredients"]) == 0

        if type_of_food_empty and ingredients_empty:
            all_hard_empty = True
        else:
            all_hard_empty = False

        
        # Find which field we're currently asking about
        current_field = None
        for key in ["type_of_food", "ingredients"]:
            if len(state["hard_constraints"][key]) == 0:
                current_field = key
                break
        
        # Build context for LLM
        field_context = ""
        if current_field and not all_hard_empty:
            field_names = {
                "type_of_food": "TYPE OF FOOD (type_of_food)",
                "ingredients": "INGREDIENTS (ingredients)"
            }
            field_context = f"\n**IMPORTANT: Currently asking about {field_names[current_field]}. ONLY update this field, KEEP all others unchanged.**\n"
        elif all_hard_empty:
            field_context = f"\n**IMPORTANT: User is providing complete information upfront. Update ALL fields mentioned in their utterance.**\n"
        
        prompt = f"""
                You are updating a conversational dialogue state JSON based on user preferences.

                User says: "{user_utterance}"
                {field_context}
                Current JSON state:
                {json.dumps(state, indent=4, ensure_ascii=False)}

                Rules:
                - **If user provides complete info in one utterance**: Update ALL mentioned fields
                - **If asking step-by-step**: ONLY update the field being asked, KEEP others unchanged
                - **Analyze carefully**: Identify which fields the user mentioned

                **Field-specific rules:**

                - **type_of_food**: 
                  + Update if user mentions food type: "món Tết", "món chính", "món phụ"
                    → Example: "món Tết" → type_of_food = ["món tết"]
                  + If NOT mentioned → KEEP as []

                - **ingredients**: 
                  + Update if user mentions ingredients
                    - Specific: "nem chua", "thịt lợn" → ingredients = ["thịt"]
                    - Any: "gì cũng được", "bất kỳ" → ingredients = ["everything"]
                  + If NOT mentioned → KEEP as []
                  
                - **cook_time**: IMPORTANT (soft_constraints)
                  + Update if user mentions time (including "don't care")
                    - "không yêu cầu", "bất kỳ", "gì cũng được" → cook_time = ["nhanh", "trung bình", "lâu"]
                    - Specific: "nhanh" → ["nhanh"], "lâu" → ["lâu"]
                  + If NOT mentioned → KEEP as []
                  
                - **algeric**: IMPORTANT (soft_constraints)
                  + Update if user mentions allergies (including "no allergies")
                    - "không dị ứng", "không bị dị ứng" → algeric = ["none"]
                    - Specific: "dị ứng tôm" → algeric = ["tôm"]
                  + If NOT mentioned → KEEP as []

                - **num_of_people**: IMPORTANT (soft_constraints)
                  + Update if user mentions servings (including "don't care")
                    - "không cần quan tâm số người" → num_of_people = ["none"]
                    - Specific: "4 người" → num_of_people = ["4"]
                  + If NOT mentioned → KEEP as []

                - **calories**: IMPORTANT (soft_constraints)
                  + Update if user mentions calories (including "don't care")
                    - "không cần quan tâm kcal" → calories = ["none"]
                    - Specific: "ít calo" → calories = ["thấp"]
                  + If NOT mentioned → KEEP as []

                **Example (Vietnamese):**

                Input: "Gợi ý món Tết, nguyên liệu nem chua, không yêu cầu thời gian nấu, tôi không bị dị ứng, không cần quan tâm số người ăn và số kcal"

                Analysis:
                - Mentions "món Tết" → Update type_of_food
                - Mentions "nem chua" → Update ingredients  
                - Mentions "không yêu cầu thời gian" → Update cook_time (all 3 types)
                - Mentions "không bị dị ứng" → Update algeric = none
                - Mentions "không cần quan tâm số người" → Update num_of_people = none
                - Mentions "không cần quan tâm kcal" → Update calories = none

                → ALL 6 fields should be updated!

                IMPORTANT: Return ONLY pure JSON, NO markdown, NO ```json, NO explanation.
                Return valid JSON with all commas and brackets in correct positions.
                """
        
        resp = genai.GenerativeModel(MODEL).generate_content(prompt)
        
        try:
            # Parse LLM response
            text = resp.text.strip()
            if "```" in text:
                parts = text.split("```")
                if len(parts) >= 2:
                    text = parts[1]
                    if text.startswith("json") or text.startswith("JSON"):
                        text = text[4:]
                    text = text.strip()
            text = text.strip()
            
            new_state = json.loads(text)
            
            # Validate structure
            if "hard_constraints" in new_state and "soft_constraints" in new_state:
                state = new_state
            else:
                print("LLM returned incomplete JSON structure, keeping old state")
                print(f"Response: {text[:200]}...")
                
        except json.JSONDecodeError as e:
            print(f"Error parsing JSON: {e}")
            print(f"Response: {resp.text[:300]}...")
            print("Keeping old state, continuing...")
        except Exception as e:
            print(f"Unexpected error: {e}")
            print(f"Response: {resp.text[:300]}...")
            print("Keeping old state, continuing...")
    
    # Handle "Accept Recommendation" intent
    if "Accept Recommendation" in intents:
        # Extract dish name from user utterance using LLM
        extract_prompt = f"""
                        User says: "{user_utterance}"

                        Extract the DISH NAME that the user is accepting/liking.
                        Return ONLY the dish name, NO explanation.
                        If no dish name found, return "NONE".

                        Examples (Vietnamese):
                        - "Tôi thích món Nem rán" → "Nem rán"
                        - "Món bánh xèo này ok" → "Bánh xèo"
                        - "Cho tôi thêm phở bò" → "Phở bò"
                        """
        dish_name = genai.GenerativeModel(MODEL).generate_content(extract_prompt).text.strip()
        
        if dish_name and dish_name != "NONE":
            if dish_name not in state["accepted_items"]:
                state["accepted_items"].append(dish_name)
                print(f"Added '{dish_name}' to accepted items")
    
    # Handle "Reject Recommendation" intent
    if "Reject Recommendation" in intents:
        # Extract dish name from user utterance using LLM
        extract_prompt = f"""
                        User says: "{user_utterance}"

                        Extract the DISH NAME that the user is rejecting/disliking.
                        Return ONLY the dish name, NO explanation.
                        If no dish name found, return "NONE".

                        Examples (Vietnamese):
                        - "Tôi không thích món Nem rán" → "Nem rán"
                        - "Món bánh xèo không hợp" → "Bánh xèo"
                        - "Bỏ phở bò đi" → "Phở bò"
                        """
        dish_name = genai.GenerativeModel(MODEL).generate_content(extract_prompt).text.strip()
        
        if dish_name and dish_name != "NONE":
            if dish_name not in state["rejected_items"]:
                state["rejected_items"].append(dish_name)
                print(f"Added '{dish_name}' to rejected items")
    
    return state

✅ State update function defined


In [6]:
## TEST STEP 2
test_user_utterance = "Tôi muốn tìm món thịt kho, dành cho 4 người, thời gian nấu trung bình"
test_intent = classify_intent(test_user_utterance)  # Expected: ["Provide Preference"]
test_state_updated = update_state(test_user_utterance, test_intent, DEFAULT_STATE)

In [7]:
test_state_updated

{'hard_constraints': {'type_of_food': [], 'ingredients': ['thịt kho']},
 'soft_constraints': {'cook_time': ['trung bình'],
  'num_of_people': ['4'],
  'calories': [],
  'algeric': []},
 'recommended_items': [],
 'accepted_items': [],
 'rejected_items': []}

In [ ]:
# STEP 3: ACTION SELECTION

def select_action(intents, state):
    """
    Select appropriate system action based on intents and current state.
    
    System Actions (RA-Rec taxonomy):
    - "Answer": Respond to user inquiry
    - "Request Information": Ask for missing constraints
    - "Info Complete": All required information collected
    
    Args:
        intents (list): Classified user intents
        state (dict): Current dialogue state
        
    Returns:
        str: Selected action
    """
    
    # If user is asking a question
    if "Inquire" in intents:
        return "Answer"
    
    # Check if all hard constraints are filled
    for key in state["hard_constraints"]:
        if len(state["hard_constraints"][key]) == 0:
            return "Request Information"
    
    # Check if soft constraints are filled
    all_soft_empty = all(len(state["soft_constraints"][key]) == 0 
                        for key in state["soft_constraints"])
    
    if all_soft_empty:
        return "Request Information"
    
    # All information collected
    return "Info Complete"

In [10]:
# Test step 3
test_action = select_action(test_intent, test_state_updated)  # Expected: "Request Information"
test_action

'Request Information'

In [ ]:
# STEP 4: RESPONSE GENERATION

def generate_response(user_utterance, action, state):
    """
    Generate appropriate system response based on selected action.
    
    Args:
        user_utterance (str): User's input text
        action (str): Selected system action
        state (dict): Current dialogue state
        
    Returns:
        str: System response
    """
    
    if action == "Request Information":
        # Find missing constraint and ask for it
        
        # Check hard constraints first
        for key in state["hard_constraints"]:
            if len(state["hard_constraints"][key]) == 0:
                questions = {
                    "type_of_food": "Bạn muốn tìm loại món gì? (ví dụ: Món Tết, món chính, món phụ...)",
                    "ingredients": "Bạn muốn món có nguyên liệu gì? (ví dụ: thịt lợn, hải sản, rau... hoặc 'gì cũng được')",
                }
                return questions.get(key, f"Could you provide information about {key}?")
        
        # Then check soft constraints
        for key in state["soft_constraints"]:
            if len(state["soft_constraints"][key]) == 0:
                questions = {
                    "cook_time": "Bạn muốn món nấu nhanh hay chậm? (nhanh: <30p, trung bình: 30-60p, lâu: >60p)",
                    "num_of_people": "Món ăn cho bao nhiêu người?",
                    "calories": "Bạn quan tâm đến mức calories không? (thấp, trung bình, cao)",
                    "algeric": "Bạn có dị ứng với thành phần nào không? (nếu không thì nói 'không')"
                }
                return questions.get(key, f"Could you provide information about {key}?")
        
        return "Tôi cần thêm thông tin để giúp bạn."
    
    elif action == "Answer":
        # Use LLM to answer user's question
        prompt = f"""
                User asks: "{user_utterance}"

                Current state:
                {json.dumps(state, indent=4, ensure_ascii=False)}

                Generate a helpful answer in Vietnamese based on the preferences and context.
                Keep it concise, friendly, and helpful.
                """
        return genai.GenerativeModel(MODEL).generate_content(prompt).text
    
    elif action == "Info Complete":
        return "Tôi đã có đầy đủ thông tin cần thiết, bạn hãy đợi tôi 1 chút nhé"
    
    return "Tôi không hiểu yêu cầu của bạn. Bạn có thể nói rõ hơn không?"

In [12]:
# Test step 4
test_generate_response = generate_response(test_user_utterance, test_action, test_state_updated)  # Expected: "Request Information"
test_generate_response

'Bạn muốn tìm loại món gì? (ví dụ: Món Tết, món chính, món phụ...)'

In [ ]:
# ============================================================
# MAIN DIALOGUE LOOP
# ============================================================

print("=" * 60)
print("RA-Rec Food Recommender System")
print("=" * 60)
print()

# Reset state at the beginning
with open(STATE_FILE, "w", encoding="utf-8") as f:
    json.dump(DEFAULT_STATE, f, indent=4, ensure_ascii=False)
print("State reset to default\n")

# Greeting
print("BOT: Xin chào! Tôi là trợ lý gợi ý món ăn.")
print("     Bạn muốn tìm món ăn gì? (ví dụ: món Tết, món nhanh, món cho 4 người...)")
print("     Type 'exit' or 'stop' to quit.\n")

# Main conversation loop
while True:
    # Get user input
    user_utterance = input("USER: ")
    
    if user_utterance.lower() in ["exit", "stop"]:
        print("\nBOT: Cảm ơn bạn đã sử dụng hệ thống. Hẹn gặp lại!")
        break
    
    # Load current state
    state = load_state()
    
    # STEP 1: Intent Classification
    intents = classify_intent(user_utterance)
    
    # Special handling: Force "Provide Preference" if we're in info-gathering mode
    asking_hard = any(len(state["hard_constraints"][key]) == 0 
                     for key in state["hard_constraints"])
    if asking_hard and "Provide Preference" not in intents:
        intents = ["Provide Preference"]
    
    # Special handling for "không" (no) responses
    if any(word in user_utterance.lower() 
           for word in ["không", "khong", "không có", "không dị ứng"]):
        # If asking about algeric and user says no
        if "algeric" in state["soft_constraints"] and state["soft_constraints"]["algeric"] == []:
            state["soft_constraints"]["algeric"] = ["none"]
            save_state(state)
            if "Provide Preference" not in intents:
                intents.append("Provide Preference")
    
    # Handle rejection of soft constraints
    if any(word in user_utterance.lower() 
           for word in ["không", "khong", "no", "skip", "nope", "don't", "dont", "bỏ qua"]):
        all_hard_filled = all(len(state["hard_constraints"][key]) > 0 
                             for key in state["hard_constraints"])
        if all_hard_filled:
            # Fill remaining soft constraints with "none"
            for key in state["soft_constraints"]:
                if len(state["soft_constraints"][key]) == 0:
                    state["soft_constraints"][key] = ["none"]
            save_state(state)
            
            action = select_action(intents, state)
            reply = generate_response(user_utterance, action, state)
            print(f"BOT: {reply}\n")
            
            if action == "Info Complete":
                print("All information saved to dialog_state.json")
                break
            continue
    
    # STEP 2: State Update
    state = update_state(user_utterance, intents, state)
    save_state(state)
    
    # STEP 3: Action Selection
    action = select_action(intents, state)
    
    # STEP 4: Response Generation
    reply = generate_response(user_utterance, action, state)
    print(f"BOT: {reply}\n")
    
    # Check if dialogue is complete
    if action == "Info Complete":
        print("All information saved to dialog_state.json")
        break

print("\n" + "=" * 60)
print("Dialogue session ended")
print("=" * 60)

RA-Rec Food Recommender System

✅ State reset to default

BOT: Xin chào! Tôi là trợ lý gợi ý món ăn.
     Bạn muốn tìm món ăn gì? (ví dụ: món Tết, món nhanh, món cho 4 người...)
     Type 'exit' or 'stop' to quit.

BOT: Tôi đã có đầy đủ thông tin cần thiết, bạn hãy đợi tôi 1 chút nhé

✅ All information saved to dialog_state.json

Dialogue session ended
